In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE

In [6]:
df = pd.read_csv('./src/ai4i2020.csv')

FileNotFoundError: [Errno 2] No such file or directory: './src/ai4i2020.csv'

In [ ]:
# Product ID 제거
df = df.drop(columns=["Product ID"])

# Type One-Hot Encoding
df = pd.get_dummies(df, columns=["Type"], drop_first=True)

# X, y 분리
X = df.drop(columns=[
    "Machine failure",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
])

y = df["Machine failure"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=49,
    stratify=y
)

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

sm = SMOTE(random_state=42)

X_train_sm, y_train_sm = sm.fit_resample(
    X_train,
    y_train
)

In [ ]:
RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42

) # 모델 만듦, balanced-> 고장 데이터가 적으니 고장데이터 가중치 높임

# rf.fit(X_train, y_train) # 학습
rf.fit(
    X_train_sm,
    y_train_sm
)

# rf_pred = rf.predict(X_test) # 예측

proba = rf.predict_proba(X_test)

rf_pred = (proba[:,1] > 0.3).astype(int)


print("="*50)
print("Random Forest")
print("="*50)

print("Accuracy :", accuracy_score(y_test, rf_pred)) # 정확도 뽑기
print()

print(classification_report(y_test, rf_pred))

ConfusionMatrixDisplay.from_predictions(y_test, rf_pred)
plt.title("Random Forest")
plt.show()

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    random_state=42,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)

print("="*50)
print("XGBoost")
print("="*50)

print("Accuracy :", accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))

ConfusionMatrixDisplay.from_predictions(y_test, xgb_pred)
plt.title("XGBoost")
plt.show()

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    random_state=42,
    n_estimators=300
)

lgbm.fit(X_train, y_train)

lgbm_pred = lgbm.predict(X_test)

print("="*50)
print("LightGBM")
print("="*50)

print("Accuracy :", accuracy_score(y_test, lgbm_pred))
print(classification_report(y_test, lgbm_pred))

ConfusionMatrixDisplay.from_predictions(y_test, lgbm_pred)
plt.title("LightGBM")
plt.show()